In [176]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms
import torch.utils.data as data

In [177]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Upload dataset, Remove nulls

In [178]:
df = pd.read_pickle('drive/MyDrive/dataset22462_augmented.pkl')
# df = df.dropna().drop(['NAME','Scale','Activity','Category','ID'], axis=1)
#convert the columns upper neck_X upper neck_Y  head top_X  head top_Y to int64
df = df.astype({'upper neck_X': 'int64', 'upper neck_Y': 'int64', 'head top_X': 'int64', 'head top_Y': 'int64'})

In [179]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22346 entries, 0 to 22345
Data columns (total 33 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   r ankle_X     22346 non-null  float64
 1   r ankle_Y     22346 non-null  float64
 2   r knee_X      22346 non-null  float64
 3   r knee_Y      22346 non-null  float64
 4   r hip_X       22346 non-null  float64
 5   r hip_Y       22346 non-null  float64
 6   l hip_X       22346 non-null  float64
 7   l hip_Y       22346 non-null  float64
 8   l knee_X      22346 non-null  float64
 9   l knee_Y      22346 non-null  float64
 10  l ankle_X     22346 non-null  float64
 11  l ankle_Y     22346 non-null  float64
 12  pelvis_X      22346 non-null  float64
 13  pelvis_Y      22346 non-null  float64
 14  thorax_X      22346 non-null  float64
 15  thorax_Y      22346 non-null  float64
 16  upper neck_X  22346 non-null  int64  
 17  upper neck_Y  22346 non-null  int64  
 18  head top_X    22346 non-nu

## Contruct the Dataset Object

In [180]:
keypoint_columns = [col for col in df.columns if col != 'image'] #todas as coluunas
keypoints = df[keypoint_columns].values.astype(np.float32)

keypoints = keypoints / 224.0 #normaliza os pontos para 0-1

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class PoseDataset(data.Dataset):
    def __init__(self, df):
        self.df = df
        self.images = df['image'].tolist()
        self.keypoints = keypoints
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        img_tensor = transform(img_array) #converter para pythorch
        kp_tensor = torch.tensor(self.keypoints[idx])
        return img_tensor, kp_tensor

## Spliting for test and train

In [181]:
dataset = PoseDataset(df)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) -train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=64, shuffle=False)

## Defining the model

In [160]:
# class BottleneckBlock(nn.Module):
#     def __init__(self, in_channels, out_channels, expansion=4):
#         super(BottleneckBlock, self).__init__()
#         bottleneck_channels = out_channels // expansion

#         self.conv1 = nn.Conv2d(in_channels, bottleneck_channels, kernel_size=1, stride=1, bias=False)
#         self.bn1 = nn.BatchNorm2d(bottleneck_channels)

#         self.conv2 = nn.Conv2d(bottleneck_channels, bottleneck_channels, kernel_size=3, stride=1, padding=1, bias=False)
#         self.bn2 = nn.BatchNorm2d(bottleneck_channels)

#         self.conv3 = nn.Conv2d(bottleneck_channels, out_channels, kernel_size=1, stride=1, bias=False)
#         self.bn3 = nn.BatchNorm2d(out_channels)

#         self.relu = nn.ReLU(inplace=True)

#         # Skip connection (identity mapping)
#         self.shortcut = nn.Sequential()
#         if in_channels != out_channels:
#             self.shortcut = nn.Sequential(
#                 nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, bias=False),
#                 nn.BatchNorm2d(out_channels)
#             )

#     def forward(self, x):
#         identity = self.shortcut(x)
#         out = self.relu(self.bn1(self.conv1(x)))
#         out = self.relu(self.bn2(self.conv2(out)))
#         out = self.bn3(self.conv3(out))
#         out += identity
#         return self.relu(out)

In [186]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Skip connection (identity mapping)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # Adding the residual connection
        out = self.relu(out)
        return out

class PoseNet(nn.Module):
    def __init__(self):
        super(PoseNet, self).__init__()
        # Using residual blocks for feature extraction
        self.conv = nn.Sequential(
            ResidualBlock(3, 32),    # Initial Conv + Residual Block
            nn.MaxPool2d(2, 2),      # 112x112

            ResidualBlock(32, 64),   # Residual Block
            nn.MaxPool2d(2, 2),      # 56x56

            ResidualBlock(64, 128),  # Residual Block
            nn.MaxPool2d(2, 2),      # 28x28

            ResidualBlock(128, 256),  # Residual Block
            nn.MaxPool2d(2, 2),      # 28x28

            ResidualBlock(256, 512), # Residual Block
            nn.AdaptiveAvgPool2d((7, 7))  # 14x14 output
        )

        self.fc = nn.Sequential(
            nn.Flatten(),

            nn.Linear(512 * 7 * 7, 1024),
            nn.ReLU(),
            nn.Dropout(0.40),

            nn.Linear(1024, 32),  # Assuming 16 keypoints, each with x, y = 32 values
            nn.Sigmoid()  # Output keypoint coordinates between [0,1]
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

## Training the model

In [187]:
model = PoseNet().to("cuda")
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0012, weight_decay=1e-5)


In [188]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MSEcriterion = nn.MSELoss()
epochs = 70
for epoch in range(epochs):
    model.train()
    train_smooth_loss, train_mse_loss = 0.0, 0.0

    for images, keypoints in train_loader:
        if device.type == 'cuda':
            images, keypoints = images.to("cuda"), keypoints.to("cuda")

        optimizer.zero_grad()
        outputs = model(images)

        smooth_loss = criterion(outputs, keypoints)
        mse_loss = MSEcriterion(outputs, keypoints)  # Just for monitoring

        smooth_loss.backward()
        optimizer.step()

        train_smooth_loss += smooth_loss.item()
        train_mse_loss += mse_loss.item()

    # scheduler.step(smooth_loss)
    avg_train_smooth_loss = train_smooth_loss / len(train_loader)
    avg_train_mse_loss = train_mse_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss (SmoothL1): {avg_train_smooth_loss:.4f}, Train Loss (MSE): {avg_train_mse_loss:.4f}")

    # Validation
    if(epoch+1) % 5 == 0:
        model.eval()
        val_smooth_loss, val_mse_loss = 0.0, 0.0
        correct, total = 0, 0  # Track total and correct keypoints

        with torch.no_grad():
            for images, keypoints in val_loader:
                if device.type == 'cuda':
                    images, keypoints = images.to("cuda"), keypoints.to("cuda")
                outputs = model(images)
                smooth_loss = criterion(outputs, keypoints)
                mse_loss = MSEcriterion(outputs, keypoints)  # Just for monitoring

                resized_keypoints = keypoints * 224
                resized_outputs = outputs * 224

                distances = torch.norm(resized_outputs.view(-1, 16, 2) - resized_keypoints.view(-1, 16, 2), dim=2)

                # Count correct keypoints (within threshold)
                correct += (distances < (0.3 * 224)).sum().item()
                total += keypoints.numel() // 2  # Number of keypoints in batch
                # print(f"Total: {total}")
                val_smooth_loss += smooth_loss.item()
                val_mse_loss += mse_loss.item()

        mAP = correct / total
        print(f"Val mAP: {correct} / {total} = {mAP:.5f}")
        avg_val_smooth_loss = val_smooth_loss / len(val_loader)
        avg_val_mse_loss = val_mse_loss / len(val_loader)
        print(f"Val Loss (SmoothL1): {avg_val_smooth_loss:.5f}, Val Loss (MSE): {avg_val_mse_loss:.5f}")

Epoch [1/70], Train Loss (SmoothL1): 0.0214, Train Loss (MSE): 0.0466
Epoch [2/70], Train Loss (SmoothL1): 0.0105, Train Loss (MSE): 0.0249
Epoch [3/70], Train Loss (SmoothL1): 0.0098, Train Loss (MSE): 0.0236
Epoch [4/70], Train Loss (SmoothL1): 0.0095, Train Loss (MSE): 0.0230
Epoch [5/70], Train Loss (SmoothL1): 0.0093, Train Loss (MSE): 0.0225
Val mAP: 65226 / 71520 = 0.91200
Val Loss (SmoothL1): 0.00843, Val Loss (MSE): 0.01730
Epoch [6/70], Train Loss (SmoothL1): 0.0091, Train Loss (MSE): 0.0221
Epoch [7/70], Train Loss (SmoothL1): 0.0090, Train Loss (MSE): 0.0218
Epoch [8/70], Train Loss (SmoothL1): 0.0088, Train Loss (MSE): 0.0216
Epoch [9/70], Train Loss (SmoothL1): 0.0087, Train Loss (MSE): 0.0212
Epoch [10/70], Train Loss (SmoothL1): 0.0085, Train Loss (MSE): 0.0209
Val mAP: 64716 / 71520 = 0.90487
Val Loss (SmoothL1): 0.00920, Val Loss (MSE): 0.01882
Epoch [11/70], Train Loss (SmoothL1): 0.0084, Train Loss (MSE): 0.0206
Epoch [12/70], Train Loss (SmoothL1): 0.0082, Train Lo

## Training logs
lr=0.01; epoch: 10; Loss: 0.0014; Val Loss: 0.0441

In [189]:
torch.save(model.state_dict(), "posev001255.pth")
